# VoiceTag — pipeline completa su ColabIndicizzazione e ricerca di conversazioni vocali mediante tag emozionali generati automaticamente.Questo notebook esegue l'intera pipeline sul corpus RAVDESS reale:1. preparazione dell'ambiente2. download di RAVDESS3. generazione della collezione di pseudo-conversazioni4. tagging acustico e creazione dell'indice5. valutazione con metriche di Information Retrieval6. figure per la relazione7. esportazione dei risultati**Runtime consigliato:** GPU T4 (non indispensabile; su CPU il tagging richiede qualche minuto in piu').

## 1. Ambiente

In [ ]:
!git clone https://github.com/Nadaa3672/voicetag.git voicetag%cd voicetag!pip install -q -r requirements.txt

In [ ]:
import sys, pathlibsys.path.insert(0, str(pathlib.Path.cwd()))from src import configprint("Tag fini :", config.EMOTIONS)print("Livelli  :", config.VALENCE)print("Attori held-out (mai visti dal tagger):", config.HELDOUT_ACTORS)

## 2. RAVDESSIl corpus contiene 1440 clip vocali di 24 attori professionisti che pronunciano due frasisemanticamente neutre in 8 emozioni. Il download e' pubblico e non richiede autenticazione.

In [ ]:
!mkdir -p data/ravdess!wget -q --show-progress -O data/ravdess.zip https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip!unzip -q -o data/ravdess.zip -d data/ravdess!find data/ravdess -name '*.wav' | wc -l

## 3. Collezione, tagging e indice`build_collection` esegue in sequenza:* **acquisizione** — indicizzazione di RAVDESS e generazione delle pseudo-conversazioni  (solo attori held-out, quindi valutazione speaker-independent);* **trasformazione** — segmentazione in finestre, tagging acustico, soft term frequency,  selezione dei tag con MMR;* **creazione indice** — document statistics, pesatura tf-idf, inversione.

In [ ]:
!python -m scripts.build_collection --n 150

In [ ]:
from src import conversations
from src.indexing import VoiceTagIndex
mf = conversations.load_manifest()
idx = VoiceTagIndex.load()
print(f"{len(mf)} conversazioni, durata media {sum(d['duration'] for d in mf)/len(mf):.1f}s, "
      f"turni medi {sum(d['n_turns'] for d in mf)/len(mf):.1f}")
print("segmenti per documento:", idx.stats()['segmenti_per_documento'])
print("df per tag:", idx.df)
print("lunghezza posting list:", idx.stats()['lunghezza_posting'])

## 4. ValutazioneI giudizi di rilevanza derivano dalla costruzione della collezione: di ogni conversazionesi conosce la traiettoria emotiva reale, quindi non serve annotazione manuale.Quattro run a confronto:| run | cosa misura ||---|---|| `canoniche_ranked` | il sistema completo su query che usano i nomi dei tag || `canoniche_boolean` | la versione ingenua: un'etichetta per conversazione, filtro senza ranking || `naturali_ranked` | query in linguaggio naturale, con espansione tramite thesaurus || `naturali_senza_espansione` | le stesse query senza espansione: il costo del vocabulary mismatch |

In [ ]:
!python -m scripts.run_eval

### Analisi di sensibilita' e ablazioni

Soglia di document frequency, potatura delle posting list e calibrazione sul prior del tagger.

In [ ]:
!python -m scripts.run_sensitivity

## 5. Figure per la relazione

In [ ]:
!python -m scripts.make_figures

In [ ]:
from IPython.display import Image, display
for name in ["fig1_architettura_idf", "fig2_similarita_tag", "fig3_confronto_run",
             "fig4_timeline", "fig5_confusione", "fig6_sensibilita"]:
    display(Image(f"results/figures/{name}.png"))

## 6. Prova interattiva del motore di ricerca

In [ ]:
from src.indexing import VoiceTagIndexfrom src import retrievalindex = VoiceTagIndex.load()for q in ["cliente arrabbiato",          "clienti insoddisfatti",          "chiamate con esperienza negativa",          "cliente tranquillo e sereno",          "chiamante in ansia per un guasto urgente"]:    out = retrieval.search(q, index, k=5)    print(f"\nQUERY: {q}")    print(f"  espansione: {out['expansion']}")    print(f"  candidati recuperati dalle posting list: {out['n_candidates']}/{index.n_docs}")    for r in out["results"]:        s, e = r["snippet_span"]        print(f"  {r['conv_id']}  score={r['score']:.3f}  tag={r['tags']}  "              f"valenza={r['valence']}  snippet={s:.1f}-{e:.1f}s")

## 7. EsportazioneL'archivio contiene indice, risultati e figure: da scaricare e committare nel repository,e da usare per compilare la relazione.

In [ ]:
!zip -qr voicetag_results.zip results data/index data/conversations/manifest.jsonfrom google.colab import filesfiles.download('voicetag_results.zip')

## 8. Interfaccia web (facoltativo)L'applicazione Streamlit si esegue in locale dopo aver scaricato l'archivio:```bashunzip voicetag_results.zipstreamlit run app/streamlit_app.py```